In [224]:
import rebound
import reboundx
from multiprocess import Pool
import numpy as np
import matplotlib.pyplot as plt
import astropy.constants as constants
import astropy.units as units

In [226]:
sim = rebound.Simulation()
sim.add('Sun', date=date, hash='sun')
sim.add('Mercury', date=date)
sim.add('Venus', date=date)
sim.add('Earth', date=date, hash='earth')
sim.add('Mars', date=date)
sim.add('Jupiter', date=date)
sim.add('Saturn', date=date)
sim.add('Uranus', date=date)
sim.add('Neptune', date=date)
sim.add('Pluto', date=date)

sim.move_to_com()
sim.convert_particle_units('AU', 'year', 'Msun')
sim.save_to_file('ss.bin')

Searching NASA Horizons for 'Sun'... 
Found: Sun (10) 
Searching NASA Horizons for 'Mercury'... 
Found: Mercury Barycenter (199) (chosen from query 'Mercury')
Searching NASA Horizons for 'Venus'... 
Found: Venus Barycenter (299) (chosen from query 'Venus')
Searching NASA Horizons for 'Earth'... 
Found: Earth-Moon Barycenter (3) (chosen from query 'Earth')
Searching NASA Horizons for 'Mars'... 
Found: Mars Barycenter (4) (chosen from query 'Mars')
Searching NASA Horizons for 'Jupiter'... 
Found: Jupiter Barycenter (5) (chosen from query 'Jupiter')
Searching NASA Horizons for 'Saturn'... 
Found: Saturn Barycenter (6) (chosen from query 'Saturn')
Searching NASA Horizons for 'Uranus'... 
Found: Uranus Barycenter (7) (chosen from query 'Uranus')
Searching NASA Horizons for 'Neptune'... 
Found: Neptune Barycenter (8) (chosen from query 'Neptune')
Searching NASA Horizons for 'Pluto'... 
Found: Pluto Barycenter (9) (chosen from query 'Pluto')


In [227]:
def simulation(par):
    num = par # unpack parameters
    # print("Starting run "+str(num)+". ")
    
    sim = rebound.Simulation('ss.bin')
    sim.integrator = "WHCKL" 
    sim.ri_whfast.safe_mode = False
    sim.ri_whfast.corrector = 17
    sim.ri_whfast.keep_unsynchronized=True
    sim.dt = 4.062/365.25
    file_name = "runs/run_"+str(num)+".bin"
    sim.save_to_file(file_name, interval=interval, delete_file=True)
    ps = sim.particles
    ps['earth'].x += par_x[num-1]

    rebx = reboundx.Extras(sim)
    gr = rebx.load_force('gr_potential')
    rebx.add_force(gr)
    gr.params['c'] = 63240 # speed of light in AU/yr
    
    cf = rebx.load_force("quadrupole")
    rebx.add_force(cf)
    
    earth_m = ps['earth'].m/1.0123000370338813
    mu_eff = f*(1*0.0123000370338813*(earth_m)**2)/(1.0123000370338813*earth_m)
    ps['earth'].params["Rcentral"] = R
    ps['earth'].params["mu_effcentral"] = mu_eff
    
    gh = rebx.load_force("gravitational_harmonics")
    rebx.add_force(gh)
    sim.particles['sun'].params["J2"] = J2
    
    ps['sun'].params["Omega"] = spin_axis_vector
    ps['sun'].params["R_eq"] = R_eq_sun
    M0 = ps['sun'].m
    
    # sim.init_megno()
    sim.exit_max_distance = 1000.
    
    try:
        for i, time in enumerate(times):
            sim.integrate(time)
            sim.particles[0].m = M0*np.exp(time / rate)
            r = (ratio-((5.14/ 1.e9)*(abs(time))))*R_e
            ps['earth'].params["Rcentral"] = r
        
        # megno = sim.megno()
        return 
        
    except rebound.Escape:
        return 1000. # At least one particle got ejected, returning large MEGNO.

In [ ]:
%%time 

with Pool() as pool:
    Ngrid = 64
    par_x = np.linspace(0.,1e-12,Ngrid)
    par_num = np.arange(1,1+Ngrid,1)
    parameters = []
    # for delx in par_x:
    for num in par_num:
        parameters.append(num)
    results = pool.map(simulation,parameters)